# 05 — Dashboard

Interactive exploration of one project's bibliometrics, geography, keywords,
network, taxonomy, and PRISMA flow — all nine required figures
(BUILD_PLAN §Stage 9), served from one `Corpus` handle so every tab is
consistent by construction (ADR 0025 Decision 7).

This is read-only: the dashboard never screens a record or writes an
override. Use `01_screen_title_abstract.ipynb` for screening.

In [ ]:
import os

import prismabib
from prismabib.errors import ConfigError
from prismabib.project import Project

print(f"prismabib {prismabib.__version__}")

# Set PRISMABIB_NOTEBOOK_SLUG to explore your own project. The default is
# the bundled reference fixture, so this notebook executes in CI with no
# Scopus key and no network access.
SLUG = os.environ.get("PRISMABIB_NOTEBOOK_SLUG", "reference")

project = None
STARTUP_ERROR = None
try:
    project = Project.open(SLUG)
    print(f"dashboard for {project.slug} at {project.root}")
except ConfigError as error:
    STARTUP_ERROR = str(error)
    print(STARTUP_ERROR)

## Build the store if it is not there yet

Layer 1 is derived, never authored (BUILD_PLAN §2.2) — rebuilding it costs
nothing but time.

In [ ]:
from prismabib.store.load import build_store

if project is not None and not project.db_path.exists():
    stats = build_store(project, rebuild=True)
    print(f"built: {stats.records_loaded} records")

## The dashboard

Sidebar: project selector, year range, taxonomy dimension, citation
snapshot. Tabs: Overview, PRISMA, Trends, Geography, Venues, Keywords,
Network, Taxonomy, Corpus browser.

In [ ]:
import panel as pn

from prismabib.viz.dashboard import dashboard

(
    dashboard(project)
    if project is not None
    else pn.pane.Markdown(
        "## Dashboard cannot start\n\n"
        f"```\n{STARTUP_ERROR}\n```\n\n"
        "This notebook defaults to the bundled `reference` fixture. To explore "
        "your own project, set the slug before launching:\n\n"
        "```bash\n"
        "PRISMABIB_NOTEBOOK_SLUG=<your-slug> \\\n"
        "  uv run panel serve notebooks/05_dashboard.ipynb --show\n"
        "```",
        styles={"color": "#8a1c1c"},
    )
)

## Then

`prismabib export` (Stage 10) writes every figure and table to `exports/`
for the manuscript — nothing in this notebook is the citable artefact.